In [14]:
import yfinance as yf
import pandas as pd

In [15]:
dat = yf.Ticker("AAPL")

# Daily prices (adjusted), then take last trading day of each month
d = dat.history(period="20y", interval="1d", auto_adjust=True)

s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end
s.index.name = "month_end"

# Make month_end a normal column and add an integer index column
monthly = s.reset_index(name="adj_close")
monthly.insert(0, "idx", range(1, len(monthly)+1))   # 1-based; use range(len(monthly)) for 0-based

#Change column names
monthly=monthly.rename(columns={"month_end":"Date", "adj_close":"Price"})


#Drop idx column
monthly=monthly.drop(columns=["idx"])

#Convert to DataFrame
microsoft_price = pd.DataFrame(monthly)

# Add column called calendar_quarter

p = microsoft_price["Date"].dt.to_period("Q")        # calendar quarters
microsoft_price["cal_year"] = p.dt.year.astype("Int64")    # 2010, 2011, ...
microsoft_price["cal_q"]    = p.dt.quarter                 # 1..4

# Final label: "Q{cal_q} {cal_year}"
microsoft_price["calendar_quarter"] = "Q" + microsoft_price["cal_q"].astype(str) + " " + microsoft_price["cal_year"].astype(str)


#Drop cal_year and cal_q column
microsoft_price=microsoft_price.drop(columns=["cal_year", "cal_q"])


microsoft_price


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_15140/2360299022.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end


,Date,Price,calendar_quarter
0,2005-08-31,1.407643,Q3 2005
1,2005-09-30,1.609379,Q3 2005
2,2005-10-31,1.728859,Q4 2005
3,2005-11-30,2.035964,Q4 2005
4,2005-12-31,2.158146,Q4 2005
...,...,...,...
236,2025-04-30,211.981125,Q2 2025
237,2025-05-31,200.622314,Q2 2025
238,2025-06-30,204.937408,Q2 2025
239,2025-07-31,207.334702,Q3 2025


In [16]:
microsoft_price.to_csv("microsoft_price.csv")

In [17]:
microsoft_price[20:]

,Date,Price,calendar_quarter
20,2007-04-30,2.996008,Q2 2007
21,2007-05-31,3.638138,Q2 2007
22,2007-06-30,3.663655,Q2 2007
23,2007-07-31,3.955451,Q3 2007
24,2007-08-31,4.157187,Q3 2007
...,...,...,...
236,2025-04-30,211.981125,Q2 2025
237,2025-05-31,200.622314,Q2 2025
238,2025-06-30,204.937408,Q2 2025
239,2025-07-31,207.334702,Q3 2025
